## Train MLP (and XGB?)

In [1]:
import sys
import pandas as pd
import plotly.express as px
import pickle
import numpy as np

sys.path.append('/home/unix/vkrhk/EmmaEmb/EmmaEmb/analysis')
from dim_reduction_utils import load_balanced_cryptic_and_regular_data
from constants import CRYPTOBENCH_TEST_DATASET, LIGYSIS_DATASET, IMG_OUTPUT_PATH, DATA_PATH, EMB_SPACES, CRYPTOBENCH_TRAIN_DATASET, SCPDB_DATASET, EMBEDDINGS_PATH

from sklearn.metrics import average_precision_score, f1_score, roc_auc_score, matthews_corrcoef, accuracy_score, hamming_loss, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.utils import shuffle
from sklearn.neural_network import MLPClassifier
from sklearn.multioutput import MultiOutputClassifier

MODELS_PATH = f'{EMBEDDINGS_PATH}/../models'

def train_test_mlp_classifier(train_emma, test_emma, emb_space_name, shuffle_ratio=None):
    X = train_emma.emb[emb_space_name]['emb']
    y = pd.get_dummies(train_emma.metadata['binding_site'], dummy_na=False).to_numpy()
    X, y = shuffle(X, y, random_state=42)

    if shuffle_ratio is not None:
        n_samples = y.shape[0]
        n_shuffle = int(n_samples * shuffle_ratio)
        y[:n_shuffle] = shuffle(y[:n_shuffle], random_state=42)
    
    # Reuse X, y from previous cells
    mlp_base = MLPClassifier(
        hidden_layer_sizes=(256, 128),
        activation="relu",
        solver="adam",
        alpha=1e-4,
        learning_rate_init=1e-3,
        max_iter=300,
        random_state=42,
    )

    mlp_clf = MultiOutputClassifier(mlp_base, n_jobs=-1)
    mlp_clf.fit(X, y, sample_weight=compute_sample_weight(class_weight="balanced", y=y))
    
    X_test = test_emma.emb[emb_space_name]['emb']
    y_test = pd.get_dummies(test_emma.metadata['binding_site'], dummy_na=False).to_numpy()
    # Reuse X_test, y_test from previous cells
    y_pred_mlp = mlp_clf.predict(X_test)

    # Convert one-hot arrays to class indices
    y_true_idx = y_test.argmax(axis=1)
    y_pred_idx = y_pred_mlp.argmax(axis=1)

    # Class names in the same order as one-hot encoding
    class_names = pd.get_dummies(test_emma.metadata["binding_site"], dummy_na=False).columns.tolist()

    # Per-class accuracy = correctly predicted samples within each true class
    cm = confusion_matrix(y_true_idx, y_pred_idx, labels=range(len(class_names)))
    per_class_accuracy = cm.diagonal() / cm.sum(axis=1)

    # Build table for display/plot
    class_acc_df = pd.DataFrame({
        "class": class_names,
        "accuracy": per_class_accuracy,
        "f1_score": [f1_score(y_test[:, i], y_pred_mlp[:, i]) for i in range(y_test.shape[1])],
        "mcc": [matthews_corrcoef(y_test[:, i], y_pred_mlp[:, i]) for i in range(y_test.shape[1])],
    })

    return class_acc_df, mlp_clf

def plot_results(results):
    feature = "binding_site"
    results_df = pd.DataFrame({
        "Embedding Space": list(results.keys()),
        "BINDING": [df[df["class"] == "BINDING"]["accuracy"].values[0] for df in results.values()],
        "CRYPTIC-BINDING": [df[df["class"] == "CRYPTIC-BINDING"]["accuracy"].values[0] for df in results.values()],
        "NON-BINDING": [df[df["class"] == "NON-BINDING"]["accuracy"].values[0] for df in results.values()]
    })
    results_df = results_df.set_index("Embedding Space").T
    fig = px.imshow(
        results_df,
        labels={
            "x": "Embedding Space",
            "y": "Feature Class",
            "color": "Mean MLP Accuracy",
        },
        title=f"Mean MLP Accuracy for {feature}",
        color_continuous_scale=[
        (0.0, "lightblue"),
        (1.0, '#303496'),],
        text_auto=".2f",
        aspect="auto",
        range_color=[0,1],  
    )

    fig.update_layout(font=dict(family="Arial"))
    fig.show()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import accuracy_score, hamming_loss
from xgboost import XGBClassifier

with open(f'{DATA_PATH}/protein_ids.pkl', 'rb') as f:
    protein_ids = pickle.load(f)

emb_space = EMB_SPACES[0]  
print(f'Testing for {emb_space[0]} embeddings ...')
emma = load_balanced_cryptic_and_regular_data(emb_space, [CRYPTOBENCH_TRAIN_DATASET, SCPDB_DATASET], DATA_PATH, protein_ids=protein_ids)

# emma.emb
# Embeddings as input features
X = emma.emb['ESM2']['emb']

# One-hot encode all metadata columns (targets)
y = pd.get_dummies(emma.metadata['binding_site'], dummy_na=False).to_numpy()

base_clf = XGBClassifier(
    n_estimators=300, # OK
    max_depth=6, # OK
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="multi:softmax",
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
    num_class=3
)

clf = MultiOutputClassifier(base_clf, n_jobs=-1)
clf.fit(X, y)

,estimator estimator: estimator objectAn estimator object implementing :term:`fit` and :term:`predict`.A :term:`predict_proba` method will be exposed only if `estimator` implementsit.,"XGBClassifier..._class=3, ...)"
,"n_jobs n_jobs: int or None, optional (default=None)The number of jobs to run in parallel.:meth:`fit`, :meth:`predict` and :meth:`partial_fit` (if supportedby the passed estimator) will be parallelized for each target.When individual estimators are fast to train or predict,using ``n_jobs > 1`` can result in slower performance dueto the parallelism overhead.``None`` means `1` unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all available processes / threads.See :term:`Glossary ` for more details... versionchanged:: 0.20 `n_jobs` default changed from `1` to `None`.",-1
,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softmax'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.9
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None


In [ ]:
from constants import CRYPTOBENCH_TEST_DATASET, LIGYSIS_DATASET
from sklearn.metrics import average_precision_score, roc_auc_score, matthews_corrcoef

emb_space = EMB_SPACES[0]  
print(f'Testing for {emb_space[0]} embeddings ...')
test_emma = load_balanced_cryptic_and_regular_data(emb_space, [CRYPTOBENCH_TEST_DATASET, LIGYSIS_DATASET], DATA_PATH)

X_test = test_emma.emb['ESM2']['emb']
y_test = pd.get_dummies(test_emma.metadata['binding_site'], dummy_na=False).to_numpy()
y_pred = clf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
hamming = hamming_loss(y_test, y_pred)
# Use probability scores for ranking metrics
y_score = np.column_stack([
    p[:, 1] if p.shape[1] > 1 else p[:, 0]
    for p in clf.predict_proba(X_test)
])

auprc = average_precision_score(y_test, y_score, average="macro")
roc_auc = roc_auc_score(y_test, y_score, average="macro")
mcc = matthews_corrcoef(y_test.argmax(axis=1), y_pred.argmax(axis=1))

print(f"AUPRC (macro): {auprc:.4f}")
print(f"ROC AUC (macro): {roc_auc:.4f}")
print(f"MCC: {mcc:.4f}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Hamming Loss: {hamming:.4f}")


Testing for ESM2 embeddings ...
812086 samples loaded.
Categories in meta data: ['binding_site']
Numerical columns in meta data: []
Embedding space 'ESM2' added successfully.
Embeddings have 1280 features each.
AUPRC (macro): 0.4475
ROC AUC (macro): 0.7975
MCC: 0.2121
Accuracy: 0.9030
Hamming Loss: 0.0623


In [43]:
from sklearn.metrics import confusion_matrix

import plotly.express as px

# Convert one-hot arrays to class indices
y_true_idx = y_test.argmax(axis=1)
y_pred_idx = y_pred.argmax(axis=1)

# Class names in the same order as one-hot encoding
class_names = pd.get_dummies(test_emma.metadata["binding_site"], dummy_na=False).columns.tolist()

# Per-class accuracy = correctly predicted samples within each true class
cm = confusion_matrix(y_true_idx, y_pred_idx, labels=range(len(class_names)))
per_class_accuracy = cm.diagonal() / cm.sum(axis=1)

# Build table for display/plot
class_acc_df = pd.DataFrame({
    "class": class_names,
    "accuracy": per_class_accuracy
})

print(class_acc_df)

fig = px.bar(
    class_acc_df,
    x="class",
    y="accuracy",
    text="accuracy",
    title="Per-class Accuracy",
    range_y=[0, 1]
)
fig.update_traces(texttemplate="%{text:.4f}", textposition="outside")
fig.update_layout(yaxis_title="Accuracy", xaxis_title="Class")
fig.show()


             class  accuracy
0          BINDING  0.065279
1  CRYPTIC-BINDING  0.029766
2      NON-BINDING  0.998045


### try MLP

In [ ]:
results = {}

with open(f'{DATA_PATH}/protein_ids.pkl', 'rb') as f:
    protein_ids = pickle.load(f)

for emb_space in EMB_SPACES[1:]:
    train_emma = load_balanced_cryptic_and_regular_data(emb_space, [CRYPTOBENCH_TRAIN_DATASET, SCPDB_DATASET], DATA_PATH, protein_ids=protein_ids)
    test_emma = load_balanced_cryptic_and_regular_data(emb_space, [CRYPTOBENCH_TEST_DATASET, LIGYSIS_DATASET], DATA_PATH)
    mlp_class_acc_df, trained_mlp = train_test_mlp_classifier(train_emma, test_emma, emb_space[0])
    results[emb_space[0]] = mlp_class_acc_df
    print(f"Results for {emb_space[0]} embeddings: ")
    print(mlp_class_acc_df)
    print()
    with open(f'{MODELS_PATH}/{emb_space[0]}.pkl','wb') as f:
        pickle.dump(trained_mlp,f)

with open(f'{IMG_OUTPUT_PATH}/accuracy/mlp_results.pkl', 'wb') as f:
    pickle.dump(results, f)

312199 samples loaded.
Categories in meta data: ['binding_site']
Numerical columns in meta data: []
Embedding space 'ProstT5' added successfully.
Embeddings have 1024 features each.
795561 samples loaded.
Categories in meta data: ['binding_site']
Numerical columns in meta data: []
Embedding space 'ProstT5' added successfully.
Embeddings have 1024 features each.
Results for ProstT5 embeddings: 
             class  accuracy  f1_score       mcc
0          BINDING  0.326195  0.220729  0.156405
1  CRYPTIC-BINDING  0.386899  0.032133  0.066538
2      NON-BINDING  0.799921  0.912447  0.256919

312199 samples loaded.
Categories in meta data: ['binding_site']
Numerical columns in meta data: []
Embedding space 'ESM1' added successfully.
Embeddings have 1280 features each.
812086 samples loaded.
Categories in meta data: ['binding_site']
Numerical columns in meta data: []
Embedding space 'ESM1' added successfully.
Embeddings have 1280 features each.
Results for ESM1 embeddings: 
             class

In [ ]:
with open('/home/unix/vkrhk/EmmaEmb/EmmaEmb/img/models/ESM2.pkl', 'rb') as f:
    loaded_mlp = pickle.load(f)


In [9]:
test_emma = load_balanced_cryptic_and_regular_data(EMB_SPACES[0], [CRYPTOBENCH_TEST_DATASET, LIGYSIS_DATASET], DATA_PATH)
X_test = test_emma.emb['ESM2']['emb']
y_test = pd.get_dummies(test_emma.metadata['binding_site'], dummy_na=False).to_numpy()
# Reuse X_test, y_test from previous cells
with open(f'{MODELS_PATH}/{EMB_SPACES[0][0]}.pkl', 'rb') as f:
    mlp_clf = pickle.load(f)
y_pred_mlp = mlp_clf.predict(X_test)

# Convert one-hot arrays to class indices
y_true_idx = y_test.argmax(axis=1)
y_pred_idx = y_pred_mlp.argmax(axis=1)

# Class names in the same order as one-hot encoding
class_names = pd.get_dummies(test_emma.metadata["binding_site"], dummy_na=False).columns.tolist()

# Per-class accuracy = correctly predicted samples within each true class
cm = confusion_matrix(y_true_idx, y_pred_idx, labels=range(len(class_names)))
per_class_accuracy = cm.diagonal() / cm.sum(axis=1)

# Build table for display/plot
class_acc_df = pd.DataFrame({
    "class": class_names,
    "accuracy": per_class_accuracy,
    "f1_score": [f1_score(y_test[:, i], y_pred_mlp[:, i]) for i in range(y_test.shape[1])],
    "mcc": [matthews_corrcoef(y_test[:, i], y_pred_mlp[:, i]) for i in range(y_test.shape[1])],
})

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred_mlp))
print(class_acc_df)

812086 samples loaded.
Categories in meta data: ['binding_site']
Numerical columns in meta data: []
Embedding space 'ESM2' added successfully.
Embeddings have 1280 features each.
              precision    recall  f1-score   support

           0       0.32      0.15      0.20     74817
           1       0.02      0.30      0.04      3158
           2       0.94      0.89      0.91    734111

   micro avg       0.86      0.82      0.84    812086
   macro avg       0.43      0.44      0.38    812086
weighted avg       0.88      0.82      0.85    812086
 samples avg       0.80      0.82      0.80    812086

             class  accuracy  f1_score       mcc
0          BINDING  0.353356  0.202238  0.165719
1  CRYPTIC-BINDING  0.286574  0.036634  0.063117
2      NON-BINDING  0.843919  0.914023  0.304266


/home/unix/vkrhk/miniconda3/envs/emmaemb/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.



In [ ]:
with open(f'{IMG_OUTPUT_PATH}/accuracy/mlp_results.pkl', 'rb') as f:
    results=pickle.load(f)

In [8]:
plot_results(results)

### Shuffle 20% of the training set
Take the training set, introduce some noise and see how it affects the MLP performance.

In [26]:
shuffled_results = {}

def train_test_mlp_classifier(train_emma, test_emma, emb_space_name, shuffle_ratio=None):
    X = train_emma.emb[emb_space_name]['emb']
    y = pd.get_dummies(train_emma.metadata['binding_site'], dummy_na=False).to_numpy()
    X, y = shuffle(X, y, random_state=42)

    if shuffle_ratio is not None:
        n_samples = y.shape[0]
        n_shuffle = int(n_samples * shuffle_ratio)
        y[:n_shuffle] = shuffle(y[:n_shuffle], random_state=42)
    
    # Reuse X, y from previous cells
    mlp_base = MLPClassifier(
        hidden_layer_sizes=(256, 128),
        activation="relu",
        solver="adam",
        alpha=1e-4,
        learning_rate_init=1e-3,
        max_iter=300,
        random_state=42
    )

    mlp_clf = MultiOutputClassifier(mlp_base, n_jobs=-1)
    mlp_clf.fit(X, y, sample_weight=compute_sample_weight(class_weight="balanced", y=y))
    
    X_test = test_emma.emb[emb_space_name]['emb']
    y_test = pd.get_dummies(test_emma.metadata['binding_site'], dummy_na=False).to_numpy()
    # Reuse X_test, y_test from previous cells
    y_pred_mlp = mlp_clf.predict(X_test)

    # Convert one-hot arrays to class indices
    y_true_idx = y_test.argmax(axis=1)
    y_pred_idx = y_pred_mlp.argmax(axis=1)

    # Class names in the same order as one-hot encoding
    class_names = pd.get_dummies(test_emma.metadata["binding_site"], dummy_na=False).columns.tolist()

    # Per-class accuracy = correctly predicted samples within each true class
    cm = confusion_matrix(y_true_idx, y_pred_idx, labels=range(len(class_names)))
    per_class_accuracy = cm.diagonal() / cm.sum(axis=1)

    # Build table for display/plot
    class_acc_df = pd.DataFrame({
        "class": class_names,
        "accuracy": per_class_accuracy,
        "f1_score": [f1_score(y_test[:, i], y_pred_mlp[:, i]) for i in range(y_test.shape[1])],
        "mcc": [matthews_corrcoef(y_test[:, i], y_pred_mlp[:, i]) for i in range(y_test.shape[1])],
    })

    return class_acc_df

shuffle_ratio = 0.2
for emb_space in EMB_SPACES:
    train_emma = load_balanced_cryptic_and_regular_data(emb_space, [CRYPTOBENCH_TRAIN_DATASET, SCPDB_DATASET], DATA_PATH, protein_ids=protein_ids)
    test_emma = load_balanced_cryptic_and_regular_data(emb_space, [CRYPTOBENCH_TEST_DATASET, LIGYSIS_DATASET], DATA_PATH)
    mlp_class_acc_df = train_test_mlp_classifier(train_emma, test_emma, emb_space[0], shuffle_ratio=shuffle_ratio)
    results[emb_space[0]] = mlp_class_acc_df
    print(f"Results for {emb_space[0]} embeddings: ")
    print(mlp_class_acc_df)
    print()

with open(f'{IMG_OUTPUT_PATH}/accuracy/mlp_results,shuffle_ratio={shuffle_ratio}.pkl', 'wb') as f:
    pickle.dump(results, f)

312199 samples loaded.
Categories in meta data: ['binding_site']
Numerical columns in meta data: []
Embedding space 'ESM2' added successfully.
Embeddings have 1280 features each.
812086 samples loaded.
Categories in meta data: ['binding_site']
Numerical columns in meta data: []
Embedding space 'ESM2' added successfully.
Embeddings have 1280 features each.
Results for ESM2 embeddings: 
             class  accuracy  f1_score       mcc
0          BINDING  0.291618  0.197184  0.139681
1  CRYPTIC-BINDING  0.332806  0.026634  0.052533
2      NON-BINDING  0.802642  0.913278  0.236943

312199 samples loaded.
Categories in meta data: ['binding_site']
Numerical columns in meta data: []
Embedding space 'ANKH' added successfully.
Embeddings have 1536 features each.
812086 samples loaded.
Categories in meta data: ['binding_site']
Numerical columns in meta data: []
Embedding space 'ANKH' added successfully.
Embeddings have 1536 features each.
Results for ANKH embeddings: 
             class  accurac

In [ ]:
plot_results(results)

### increase shuffle
Increasingly shuffle the training set and observe how the accurancy changes.

In [ ]:
results = {}
for shuffle_ratio in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:
    emb_space = EMB_SPACES[0]
    train_emma = load_balanced_cryptic_and_regular_data(emb_space, [CRYPTOBENCH_TRAIN_DATASET, SCPDB_DATASET], DATA_PATH, protein_ids=protein_ids)
    test_emma = load_balanced_cryptic_and_regular_data(emb_space, [CRYPTOBENCH_TEST_DATASET, LIGYSIS_DATASET], DATA_PATH)
    mlp_class_acc_df = train_test_mlp_classifier(train_emma, test_emma, emb_space[0], shuffle_ratio=shuffle_ratio)
    results[emb_space[0]] = mlp_class_acc_df
    print(f"Results for {emb_space[0]} embeddings: ")
    print(mlp_class_acc_df)
    print()

with open(f'{IMG_OUTPUT_PATH}/accuracy/mlp_results,various-shuffles.pkl', 'wb') as f:
    pickle.dump(results, f)


### Try pyTorch

In [2]:
import csv
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.utils import class_weight
import numpy as np
from sklearn import metrics

DROPOUT = 0.3
LAYER_WIDTH = 1024

class EmmaClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.layer_1 = nn.Linear(in_features=input_dim, out_features=LAYER_WIDTH)
        self.dropout1 = nn.Dropout(DROPOUT)

        self.layer_2 = nn.Linear(in_features=LAYER_WIDTH, out_features=LAYER_WIDTH)
        self.dropout2 = nn.Dropout(DROPOUT)

        self.layer_3 = nn.Linear(in_features=LAYER_WIDTH, out_features=LAYER_WIDTH)
        self.dropout3 = nn.Dropout(DROPOUT)

        self.layer_4 = nn.Linear(in_features=LAYER_WIDTH, out_features=3)

        self.relu = nn.ReLU()

    def forward(self, x):
      # Intersperse the ReLU activation function between layers
       return self.layer_4(self.dropout3(self.relu(self.layer_3(self.dropout2(self.relu(self.layer_2(self.dropout1(self.relu(self.layer_1(x))))))))))


class EmmaDataset(Dataset):
    def __init__(self, emma, emb_space_name):
        self.X = emma.emb[emb_space_name]['emb']
        self.y = np.column_stack((
            (emma.metadata['binding_site'] == 'BINDING').to_numpy().astype(int),
            (emma.metadata['binding_site'] == 'CRYPTIC-BINDING').to_numpy().astype(int),
            (emma.metadata['binding_site'] == 'NON-BINDING').to_numpy().astype(int)))

        self.X, self.y = shuffle(self.X, self.y, random_state=42)

        self.Xs = torch.tensor(self.X, dtype=torch.float32)
        self.Ys = torch.tensor(self.y, dtype=torch.int64)

    def __len__(self):
        assert len(self.Xs) == len(self.Ys)
        return len(self.Xs)

    def __getitem__(self, idx):
        x = self.Xs[idx]
        y = self.Ys[idx]
        return x, y

def compute_weights(y):
    counts = y.sum(axis=0)

    N = y.shape[0]
    C = y.shape[1]

    weights = N / (C * counts)
    return weights

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Running on device {device} ...')

with open(f'{DATA_PATH}/protein_ids.pkl', 'rb') as f:
    protein_ids = pickle.load(f)



Running on device cuda ...


In [40]:
sum((train_emma.metadata['binding_site'] == 'NON-BINDING').to_numpy().astype(int))

np.int64(1792157)

In [3]:
emb_space = EMB_SPACES[0]
train_emma = load_balanced_cryptic_and_regular_data(emb_space, [CRYPTOBENCH_TRAIN_DATASET, SCPDB_DATASET], DATA_PATH, protein_ids=None)
test_emma = load_balanced_cryptic_and_regular_data(emb_space, [CRYPTOBENCH_TEST_DATASET, LIGYSIS_DATASET], DATA_PATH)

train_dataset = EmmaDataset(train_emma, emb_space[0])
test_dataset = EmmaDataset(test_emma, emb_space[0])


2042764 samples loaded.
Categories in meta data: ['binding_site']
Numerical columns in meta data: []
Embedding space 'ESM2' added successfully.
Embeddings have 1280 features each.
812086 samples loaded.
Categories in meta data: ['binding_site']
Numerical columns in meta data: []
Embedding space 'ESM2' added successfully.
Embeddings have 1280 features each.


In [7]:
from torch.utils.data import WeightedRandomSampler

SAMPLER = True
epochs = 30
batch_size = 2048

def train(emb_space, train_protein_ids=protein_ids):
    # train_emma = load_balanced_cryptic_and_regular_data(emb_space, [CRYPTOBENCH_TRAIN_DATASET, SCPDB_DATASET], DATA_PATH, protein_ids=train_protein_ids)
    # test_emma = load_balanced_cryptic_and_regular_data(emb_space, [CRYPTOBENCH_TEST_DATASET, LIGYSIS_DATASET], DATA_PATH)

    train_dataset = EmmaDataset(train_emma, emb_space[0])
    test_dataset = EmmaDataset(test_emma, emb_space[0])

    # Build per-sample weights from one-hot labels in train_dataset
    _, y_train_sampler = train_dataset[:]  # shape: [N, C]
    class_counts = y_train_sampler.sum(dim=0).float()
    inv_class_weights = 1.0 / class_counts
    sample_weights = (y_train_sampler.float() * inv_class_weights).sum(dim=1)

    weighted_sampler = WeightedRandomSampler(
        weights=sample_weights.cpu(),
        num_samples=len(sample_weights),
        replacement=True
    )

    class FocalLoss(nn.Module):
        def __init__(self, alpha=None, gamma=2):
            super().__init__()
            self.alpha = alpha
            self.gamma = gamma

        def forward(self, logits, targets):
            ce_loss = nn.functional.cross_entropy(
                logits, targets, weight=self.alpha, reduction='none'
            )
            pt = torch.exp(-ce_loss)
            focal_loss = (1 - pt) ** self.gamma * ce_loss
            return focal_loss.mean()

    model = EmmaClassifier(input_dim=train_dataset.X.shape[1]).to(device)
    optimizer = torch.optim.AdamW(params=model.parameters(),
                                lr=0.00001)

    _, y_train = train_dataset[:]
    X_test, y_test = test_dataset[:]

    class_weights = compute_weights(y_train).to(device)
    # criterion = FocalLoss(gamma=2)
    criterion = nn.CrossEntropyLoss() #weight=class_weights)

    X_test, y_test = X_test.to(device), y_test.to(device).float()

    train_losses, test_losses = [], []

    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, sampler=weighted_sampler if SAMPLER else None)

    best_loss = float('inf')
    for epoch in range(epochs):
        #
        # TEST
        #
        model.eval()
        with torch.inference_mode():

            test_logits = model(X_test).squeeze()
            test_probabilities = torch.softmax(test_logits, dim=1)

            test_loss = criterion(test_logits, torch.argmax(y_test, dim=1))
            y_true = torch.argmax(y_test, dim=1).cpu().numpy()
            y_pred = torch.argmax(test_probabilities, dim=1).cpu().numpy()

            cm = confusion_matrix(y_true, y_pred, labels=np.arange(y_test.shape[1]))
            print("Confusion matrix:")
            print(cm)
            per_class_acc = np.divide(
                cm.diagonal(),
                cm.sum(axis=1),
                out=np.zeros(cm.shape[0], dtype=float),
                where=cm.sum(axis=1) != 0
            )

            class_names = pd.get_dummies(test_emma.metadata["binding_site"], dummy_na=False).columns.tolist()
            class_acc_df = pd.DataFrame({"class": class_names, "accuracy": per_class_acc})

            test_acc = accuracy_score(y_true, y_pred)
            test_mcc = matthews_corrcoef(y_true, y_pred)
            test_f1 = f1_score(y_true, y_pred, average='macro')
            test_auprc = average_precision_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy())
            test_roc_auc = roc_auc_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy())

            if test_loss < best_loss:
                best_loss = test_loss
                import copy
                best_model = copy.deepcopy(model.state_dict())

        #
        # TRAIN
        #
        model.train()
        batch_losses = []
        for x_batch, y_batch in train_dataloader:

            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            y_logits = model(x_batch)

            y_batch = torch.argmax(y_batch, dim=1)
            loss = criterion(y_logits,
                            y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            batch_losses.append(loss.cpu().detach().numpy())

        train_losses.append(sum(batch_losses) / len(batch_losses))

        print(f"Epoch: {epoch} | Loss: {loss:.5f}, Accuracy: {(test_acc * 100):.2f}% | Test loss: {test_loss:.5f}, AUC: {test_roc_auc:.4f}, MCC: {test_mcc:.4f}, F1: {test_f1:.4f}, AUPRC: {test_auprc:.4f}")    
        print(f"Class-wise accuracy:\n\t{class_acc_df}\n")
        print()

    print("Best model performance:")
    model.eval()
    with torch.inference_mode():

        test_logits = model(X_test).squeeze()
        test_probabilities = torch.softmax(test_logits, dim=1)

        test_loss = criterion(test_logits, torch.argmax(y_test, dim=1))
        y_true = torch.argmax(y_test, dim=1).cpu().numpy()
        y_pred = torch.argmax(test_probabilities, dim=1).cpu().numpy()

        cm = confusion_matrix(y_true, y_pred, labels=np.arange(y_test.shape[1]))
        print("Confusion matrix:")
        print(cm)
        per_class_acc = np.divide(
            cm.diagonal(),
            cm.sum(axis=1),
            out=np.zeros(cm.shape[0], dtype=float),
            where=cm.sum(axis=1) != 0
        )

        class_names = pd.get_dummies(test_emma.metadata["binding_site"], dummy_na=False).columns.tolist()
        class_acc_df = pd.DataFrame({"class": class_names, "accuracy": per_class_acc})

        test_acc = accuracy_score(y_true, y_pred)
        test_mcc = matthews_corrcoef(y_true, y_pred)
    print(f"Test loss: {test_loss:.5f}, Accuracy: {(test_acc * 100):.2f}%, AUC: {test_roc_auc:.4f}, MCC: {test_mcc:.4f}, F1: {test_f1:.4f}, AUPRC: {test_auprc:.4f}")
    # with open(f'{MODELS_PATH}/{emb_space[0]}_torch_best_model.pth', 'wb') as f:
    #     torch.save(best_model, f)
    # import pickle
    # with open(f'{IMG_OUTPUT_PATH}/performance/{emb_space[0]}_torch.pkl', 'wb') as f:
    #     pickle.dump({
    #         "test_acc": test_acc,
    #         "test_roc_auc": test_roc_auc,
    #         "test_mcc": test_mcc,
    #         "test_f1": test_f1,
    #         "test_auprc": test_auprc,
    #         "class_acc_df": class_acc_df
    #     }, f)

def test(emb_space, model):
    test_emma = load_balanced_cryptic_and_regular_data(emb_space, [CRYPTOBENCH_TEST_DATASET, LIGYSIS_DATASET], DATA_PATH)

    test_dataset = EmmaDataset(test_emma, emb_space[0])
    X_test, y_test = test_dataset[:]

    X_test, y_test = X_test.to(device), y_test.to(device).float()

    model.eval()
    with torch.inference_mode():

        test_logits = model(X_test).squeeze()
        test_probabilities = torch.softmax(test_logits, dim=1)

        y_true = torch.argmax(y_test, dim=1).cpu().numpy()
        y_pred = torch.argmax(test_probabilities, dim=1).cpu().numpy()

        cm = confusion_matrix(y_true, y_pred, labels=np.arange(y_test.shape[1]))
        per_class_acc = np.divide(
            cm.diagonal(),
            cm.sum(axis=1),
            out=np.zeros(cm.shape[0], dtype=float),
            where=cm.sum(axis=1) != 0
        )

        class_names = pd.get_dummies(test_emma.metadata["binding_site"], dummy_na=False).columns.tolist()
        class_acc_df = pd.DataFrame({"class": class_names, "accuracy": per_class_acc})

        test_acc = accuracy_score(y_true, y_pred)
        test_mcc = matthews_corrcoef(y_true, y_pred)
        test_f1 = f1_score(y_true, y_pred, average='macro')
        test_auprc = average_precision_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy())
        test_roc_auc = roc_auc_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy())
    print(f' Confusion matrix:')
    print(cm)
    print(f'Accuracy {(test_acc * 100):.2f}%, AUC: {test_roc_auc:.4f}, MCC: {test_mcc:.4f}, F1: {test_f1:.4f}, AUPRC: {test_auprc:.4f}')

In [ ]:
def calculate_class_weights(y):
    unique_classes, class_counts = np.unique(y, return_counts=True)
    total_samples = len(y)
    class_weights = np.zeros(len(unique_classes))

    for class_label, class_count in zip(unique_classes, class_counts):
        class_weight = total_samples / (2.0 * class_count)
        class_weights[class_label] = class_weight

    return class_weights

torch.tensor(calculate_class_weights(np.argmax(y_train, axis=1))).to(device)

[0 1 2]


tensor([ 4.3034, 77.0099,  0.5699], device='cuda:0', dtype=torch.float64)

In [6]:
emb_space = EMB_SPACES[0]
batch_size = 2048

train_emma = load_balanced_cryptic_and_regular_data(emb_space, [CRYPTOBENCH_TRAIN_DATASET, SCPDB_DATASET], DATA_PATH, protein_ids=None)
test_emma = load_balanced_cryptic_and_regular_data(emb_space, [CRYPTOBENCH_TEST_DATASET, LIGYSIS_DATASET], DATA_PATH)

train_dataset = EmmaDataset(train_emma, emb_space[0])
test_dataset = EmmaDataset(test_emma, emb_space[0])

_, y_train = train_dataset[:]
X_test, y_test = test_dataset[:]

X_test, y_test = X_test.to(device), y_test.to(device).float()
train_losses, test_losses = [], []
train_dataloader = DataLoader(train_dataset, batch_size=batch_size)


2042764 samples loaded.
Categories in meta data: ['binding_site']
Numerical columns in meta data: []
Embedding space 'ESM2' added successfully.
Embeddings have 1280 features each.
812086 samples loaded.
Categories in meta data: ['binding_site']
Numerical columns in meta data: []
Embedding space 'ESM2' added successfully.
Embeddings have 1280 features each.


In [ ]:
SAMPLER = False
epochs = 20
batch_size = 4096
train_dataloader = DataLoader(train_dataset, batch_size=batch_size)

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce_loss = nn.functional.cross_entropy(
            logits, targets, weight=self.alpha, reduction='none'
        )
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()


model = EmmaClassifier(input_dim=train_dataset.X.shape[1]).to(device)
optimizer = torch.optim.AdamW(params=model.parameters(),
                            lr=0.00001)


## lr=0.00001, gamma=1.5, alpha=[2, 7, 0.5]: 
# Epoch: 7 | Loss: 0.17398, Accuracy: 69.48% | Test loss: 0.30095, AUC: 0.6743, MCC: 0.2080, F1: 0.3737, AUPRC: 0.3971

## lr=0.000005, gamma=1.5, alpha=[2, 8, 0.5]: 
# Epoch: 13 | Loss: 0.19552, Accuracy: 68.01% | Test loss: 0.31222, AUC: 0.6728, MCC: 0.1986, F1: 0.3669, AUPRC: 0.3941

# class_weights = torch.tensor(calculate_class_weights(np.argmax(y_train, axis=1))).float().to(device)
criterion = FocalLoss(alpha=torch.tensor([2, 7, 0.5]).float().to(device), gamma=1.5)
# criterion = FocalLoss(alpha=None, gamma=2)
# criterion = nn.CrossEntropyLoss(weight=class_weights)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size)

best_loss = float('inf')
for epoch in range(epochs):
    #
    # TEST
    #
    model.eval()
    with torch.inference_mode():

        test_logits = model(X_test).squeeze()
        test_probabilities = torch.softmax(test_logits, dim=1)

        test_loss = criterion(test_logits, torch.argmax(y_test, dim=1))
        y_true = torch.argmax(y_test, dim=1).cpu().numpy()
        y_pred = torch.argmax(test_probabilities, dim=1).cpu().numpy()

        cm = confusion_matrix(y_true, y_pred, labels=np.arange(y_test.shape[1]))
        print("Confusion matrix:")
        print(cm)
        per_class_acc = np.divide(
            cm.diagonal(),
            cm.sum(axis=1),
            out=np.zeros(cm.shape[0], dtype=float),
            where=cm.sum(axis=1) != 0
        )

        class_names = pd.get_dummies(test_emma.metadata["binding_site"], dummy_na=False).columns.tolist()
        class_acc_df = pd.DataFrame({"class": class_names, "accuracy": per_class_acc})

        test_acc = accuracy_score(y_true, y_pred)
        test_mcc = matthews_corrcoef(y_true, y_pred)
        test_f1 = f1_score(y_true, y_pred, average='macro')
        test_auprc = average_precision_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy())
        test_roc_auc = roc_auc_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy())

        if test_loss < best_loss:
            best_loss = test_loss
            import copy
            best_model = copy.deepcopy(model.state_dict())

    #
    # TRAIN
    #
    model.train()
    batch_losses = []
    for x_batch, y_batch in train_dataloader:

        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        y_logits = model(x_batch)

        y_batch = torch.argmax(y_batch, dim=1)
        loss = criterion(y_logits,
                        y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_losses.append(loss.cpu().detach().numpy())

    train_losses.append(sum(batch_losses) / len(batch_losses))

    print(f"Epoch: {epoch} | Loss: {loss:.5f}, Accuracy: {(test_acc * 100):.2f}% | Test loss: {test_loss:.5f}, AUC: {test_roc_auc:.4f}, MCC: {test_mcc:.4f}, F1: {test_f1:.4f}, AUPRC: {test_auprc:.4f}")    
    print(f"Class-wise accuracy:\n\t{class_acc_df}\n")
    print()

print("Best model performance:")
model.eval()
with torch.inference_mode():

    test_logits = model(X_test).squeeze()
    test_probabilities = torch.softmax(test_logits, dim=1)

    test_loss = criterion(test_logits, torch.argmax(y_test, dim=1))
    y_true = torch.argmax(y_test, dim=1).cpu().numpy()
    y_pred = torch.argmax(test_probabilities, dim=1).cpu().numpy()

    cm = confusion_matrix(y_true, y_pred, labels=np.arange(y_test.shape[1]))
    print("Confusion matrix:")
    print(cm)
    per_class_acc = np.divide(
        cm.diagonal(),
        cm.sum(axis=1),
        out=np.zeros(cm.shape[0], dtype=float),
        where=cm.sum(axis=1) != 0
    )

    class_names = pd.get_dummies(test_emma.metadata["binding_site"], dummy_na=False).columns.tolist()
    class_acc_df = pd.DataFrame({"class": class_names, "accuracy": per_class_acc})

    test_acc = accuracy_score(y_true, y_pred)
    test_mcc = matthews_corrcoef(y_true, y_pred)
print(f"Test loss: {test_loss:.5f}, Accuracy: {(test_acc * 100):.2f}%, AUC: {test_roc_auc:.4f}, MCC: {test_mcc:.4f}, F1: {test_f1:.4f}, AUPRC: {test_auprc:.4f}")


Confusion matrix:
[[     0    230  74587]
 [     0      7   3151]
 [     0   1228 732883]]
Epoch: 0 | Loss: 0.24412, Accuracy: 90.25% | Test loss: 0.33553, AUC: 0.4945, MCC: 0.0048, F1: 0.3173, AUPRC: 0.3338
Class-wise accuracy:
	             class  accuracy
0          BINDING  0.000000
1  CRYPTIC-BINDING  0.002217
2      NON-BINDING  0.998327


Confusion matrix:
[[ 59212      0  15605]
 [  2635      0    523]
 [272090      0 462021]]
Epoch: 1 | Loss: 0.22360, Accuracy: 64.18% | Test loss: 0.21297, AUC: 0.7635, MCC: 0.2468, F1: 0.3507, AUPRC: 0.4253
Class-wise accuracy:
	             class  accuracy
0          BINDING  0.791424
1  CRYPTIC-BINDING  0.000000
2      NON-BINDING  0.629361


Confusion matrix:
[[ 58431      3  16383]
 [  2656      0    502]
 [254401      2 479708]]
Epoch: 2 | Loss: 0.21186, Accuracy: 66.27% | Test loss: 0.22121, AUC: 0.7591, MCC: 0.2574, F1: 0.3597, AUPRC: 0.4133
Class-wise accuracy:
	             class  accuracy
0          BINDING  0.780986
1  CRYPTIC-BINDI

KeyboardInterrupt: 

In [23]:
def train2(emb_space, epochs=10, batch_size=4096):
        
    train_emma = load_balanced_cryptic_and_regular_data(emb_space, [CRYPTOBENCH_TRAIN_DATASET, SCPDB_DATASET], DATA_PATH, protein_ids=None)
    test_emma = load_balanced_cryptic_and_regular_data(emb_space, [CRYPTOBENCH_TEST_DATASET, LIGYSIS_DATASET], DATA_PATH)

    train_dataset = EmmaDataset(train_emma, emb_space[0])
    test_dataset = EmmaDataset(test_emma, emb_space[0])

    X_test, y_test = test_dataset[:]

    X_test, y_test = X_test.to(device), y_test.to(device).float()
    train_losses, test_losses = [], []
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size)

    class FocalLoss(nn.Module):
        def __init__(self, alpha=None, gamma=2):
            super().__init__()
            self.alpha = alpha
            self.gamma = gamma

        def forward(self, logits, targets):
            ce_loss = nn.functional.cross_entropy(
                logits, targets, weight=self.alpha, reduction='none'
            )
            pt = torch.exp(-ce_loss)
            focal_loss = (1 - pt) ** self.gamma * ce_loss
            return focal_loss.mean()


    model = EmmaClassifier(input_dim=train_dataset.X.shape[1]).to(device)
    optimizer = torch.optim.AdamW(params=model.parameters(),
                                lr=0.00001)
    
    criterion = FocalLoss(alpha=torch.tensor([2, 7, 0.5]).float().to(device), gamma=1.5)
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size)

    for epoch in range(epochs):
        #
        # TEST
        #
        model.eval()
        with torch.inference_mode():

            test_logits = model(X_test).squeeze()
            test_probabilities = torch.softmax(test_logits, dim=1)

            test_loss = criterion(test_logits, torch.argmax(y_test, dim=1))
            y_true = torch.argmax(y_test, dim=1).cpu().numpy()
            y_pred = torch.argmax(test_probabilities, dim=1).cpu().numpy()

            cm = confusion_matrix(y_true, y_pred, labels=np.arange(y_test.shape[1]))
            print("Confusion matrix:")
            print(cm)
            per_class_acc = np.divide(
                cm.diagonal(),
                cm.sum(axis=1),
                out=np.zeros(cm.shape[0], dtype=float),
                where=cm.sum(axis=1) != 0
            )

            class_names = pd.get_dummies(test_emma.metadata["binding_site"], dummy_na=False).columns.tolist()
            class_acc_df = pd.DataFrame({"class": class_names, "accuracy": per_class_acc})

            test_acc = accuracy_score(y_true, y_pred)
            test_mcc = matthews_corrcoef(y_true, y_pred)
            test_f1_weighted = f1_score(y_true, y_pred, average='weighted')
            test_f1_macro = f1_score(y_true, y_pred, average='macro')
            test_f1_micro = f1_score(y_true, y_pred, average='micro')
            test_f1 = test_f1_macro
            test_auprc = average_precision_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy())
            test_roc_auc = roc_auc_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy())

        #
        # TRAIN
        #
        model.train()
        batch_losses = []
        for x_batch, y_batch in train_dataloader:

            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            y_logits = model(x_batch)

            y_batch = torch.argmax(y_batch, dim=1)
            loss = criterion(y_logits,
                            y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            batch_losses.append(loss.cpu().detach().numpy())

        train_losses.append(sum(batch_losses) / len(batch_losses))

        print(f"Epoch: {epoch} | Loss: {loss:.5f}, Accuracy: {(test_acc * 100):.2f}% | Test loss: {test_loss:.5f}, AUC: {test_roc_auc:.4f}, MCC: {test_mcc:.4f}, F1: {test_f1:.4f}, AUPRC: {test_auprc:.4f}")    
        print(f"Class-wise accuracy:\n\t{class_acc_df}\n")
        print()

    print("Best model performance:")
    model.eval()
    with torch.inference_mode():

        test_logits = model(X_test).squeeze()
        test_probabilities = torch.softmax(test_logits, dim=1)

        test_loss = criterion(test_logits, torch.argmax(y_test, dim=1))
        y_true = torch.argmax(y_test, dim=1).cpu().numpy()
        y_pred = torch.argmax(test_probabilities, dim=1).cpu().numpy()

        cm = confusion_matrix(y_true, y_pred, labels=np.arange(y_test.shape[1]))
        print("Confusion matrix:")
        print(cm)
        per_class_acc = np.divide(
            cm.diagonal(),
            cm.sum(axis=1),
            out=np.zeros(cm.shape[0], dtype=float),
            where=cm.sum(axis=1) != 0
        )

        class_names = pd.get_dummies(test_emma.metadata["binding_site"], dummy_na=False).columns.tolist()
        class_acc_df = pd.DataFrame({"class": class_names, "accuracy": per_class_acc})

        test_acc = accuracy_score(y_true, y_pred)
        test_mcc = matthews_corrcoef(y_true, y_pred)
    print(f"Test loss: {test_loss:.5f}, Accuracy: {(test_acc * 100):.2f}%, AUC: {test_roc_auc:.4f}, MCC: {test_mcc:.4f}, F1: {test_f1:.4f}, AUPRC: {test_auprc:.4f}")
    ## lr=0.00001, gamma=1.5, alpha=[2, 7, 0.5]: 
    # Epoch: 7 | Loss: 0.17398, Accuracy: 69.48% | Test loss: 0.30095, AUC: 0.6743, MCC: 0.2080, F1: 0.3737, AUPRC: 0.3971
    
    ## lr=0.000005, gamma=1.5, alpha=[2, 8, 0.5]: 
    # Epoch: 13 | Loss: 0.19552, Accuracy: 68.01% | Test loss: 0.31222, AUC: 0.6728, MCC: 0.1986, F1: 0.3669, AUPRC: 0.3941
    with open(f'{MODELS_PATH}/{emb_space[0]}_train2_torch_best_model.pth', 'wb') as f:
        torch.save(model, f)
    import pickle
    with open(f'{IMG_OUTPUT_PATH}/performance/train2-epochs={epochs}/{emb_space[0]}_torch.pkl', 'wb') as f:
        pickle.dump({
            "test_acc": test_acc,
            "test_roc_auc": test_roc_auc,
            "test_mcc": test_mcc,
            "test_f1_weighted": test_f1_weighted,
            "test_f1_macro": test_f1_macro,
            "test_f1_micro": test_f1_micro,
            "test_auprc": test_auprc,
            "class_acc_df": class_acc_df,
            "cm": cm
        }, f)


In [24]:
for emb_space in EMB_SPACES:
    train2(emb_space)

2042764 samples loaded.
Categories in meta data: ['binding_site']
Numerical columns in meta data: []
Embedding space 'ESM2' added successfully.
Embeddings have 1280 features each.
812086 samples loaded.
Categories in meta data: ['binding_site']
Numerical columns in meta data: []
Embedding space 'ESM2' added successfully.
Embeddings have 1280 features each.
Confusion matrix:
[[    26  74541    250]
 [     2   3150      6]
 [   205 732288   1618]]
Epoch: 0 | Loss: 0.24442, Accuracy: 0.59% | Test loss: 0.33949, AUC: 0.4630, MCC: -0.0029, F1: 0.0043, AUPRC: 0.3240
Class-wise accuracy:
	             class  accuracy
0          BINDING  0.000348
1  CRYPTIC-BINDING  0.997467
2      NON-BINDING  0.002204


Confusion matrix:
[[ 59380      0  15437]
 [  2652      0    506]
 [278583      0 455528]]
Epoch: 1 | Loss: 0.22032, Accuracy: 63.41% | Test loss: 0.21323, AUC: 0.7625, MCC: 0.2423, F1: 0.3472, AUPRC: 0.4249
Class-wise accuracy:
	             class  accuracy
0          BINDING  0.793670
1  CR

In [81]:
SAMPLER = True
for emb_space in EMB_SPACES[0:1]:
    print(f"Training MLP on {emb_space[0]} embeddings without weighted sampler...")
    train(emb_space, train_protein_ids=None)

Training MLP on ESM2 embeddings without weighted sampler...
Confusion matrix:
[[  4425      8  70384]
 [   211      1   2946]
 [ 17231     54 716826]]
Epoch: 0 | Loss: 0.82364, Accuracy: 88.81% | Test loss: 1.08310, AUC: 0.5347, MCC: 0.0637, F1: 0.3442, AUPRC: 0.3503
Class-wise accuracy:
	             class  accuracy
0          BINDING  0.059144
1  CRYPTIC-BINDING  0.000317
2      NON-BINDING  0.976455


Confusion matrix:
[[  7443  56981  10393]
 [   791   1911    456]
 [ 38559 339226 356326]]
Epoch: 1 | Loss: 0.78157, Accuracy: 45.03% | Test loss: 1.24868, AUC: 0.6663, MCC: 0.1108, F1: 0.2597, AUPRC: 0.3706
Class-wise accuracy:
	             class  accuracy
0          BINDING  0.099483
1  CRYPTIC-BINDING  0.605130
2      NON-BINDING  0.485384


Confusion matrix:
[[  8678  56647   9492]
 [   818   1947    393]
 [ 37386 360007 336718]]
Epoch: 2 | Loss: 0.75170, Accuracy: 42.77% | Test loss: 1.28408, AUC: 0.6591, MCC: 0.1118, F1: 0.2583, AUPRC: 0.3740
Class-wise accuracy:
	             c

### Use filtered scPDB subset
Use whole scPDB that doesn't leak into the test set.

In [ ]:
for emb_space in EMB_SPACES:
    print(f"Training MLP on {emb_space[0]} embeddings with weighted sampler...")
    train(emb_space, train_protein_ids=None)

In [ ]:
emb_space = EMB_SPACES[1]
with open(f'{MODELS_PATH}/{emb_space[0]}_torch_best_model.pth', 'rb') as f:
    model = EmmaClassifier(input_dim=1536).to(device)
    model.load_state_dict(torch.load(f))

test(emb_space, model)

# ANKH
#  Confusion matrix:
# [[ 21964  25586  25992]
#  [  1262   1102    735]
#  [ 71575  91633 555712]]
# Accuracy 72.75%, AUC: 0.6794, MCC: 0.1987, F1: 0.3777, AUPRC: 0.3902

# PROTT5
# Confusion matrix:
# [[ 18085  28561  28171]
#  [  1467   1011    680]
#  [ 50689  93302 590120]]
# Accuracy: 75.02%, AUC: 0.6692, MCC: 0.2021, F1: 0.3779, AUPRC: 0.3875

# ESM2
#  Confusion matrix:
#     R       C      N
# [[ 20712  22963  31142]  R
#  [  1531    763    864]  C
#  [ 56553  71847 605711]] N
# Accuracy 77.23%, AUC: 0.6563, MCC: 0.2117, F1: 0.3894, AUPRC: 0.3937

# PROSTT5
# Confusion matrix:
# [[ 21340  28829  24648]
#  [  1439   1070    649]
#  [ 64910  94433 574768]]
# Accuracy 73.54%, AUC: 0.6792, MCC: 0.2104, F1: 0.3803, AUPRC: 0.3902


812086 samples loaded.
Categories in meta data: ['binding_site']
Numerical columns in meta data: []
Embedding space 'ANKH' added successfully.
Embeddings have 1536 features each.
 Confusion matrix:
[[ 21340  28829  24648]
 [  1439   1070    649]
 [ 64910  94433 574768]]
Accuracy 73.54%, AUC: 0.6792, MCC: 0.2104, F1: 0.3803, AUPRC: 0.3902


In [ ]:
class Stage1Dataset(Dataset):
    def __init__(self, base_dataset):
        self.X = base_dataset.Xs
        y = base_dataset.Ys  # one-hot [N,3]

        # 1 = binding or cryptic, 0 = non-binding
        y_binary = (y[:, 0] + y[:, 1]) > 0
        self.Ys = y_binary.long()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Ys[idx]


class Stage2Dataset(Dataset):
    def __init__(self, base_dataset):
        self.X = base_dataset.Xs
        y = base_dataset.Ys  # one-hot [N,3]

        # keep only binding + cryptic
        mask = (y[:, 0] + y[:, 1]) > 0

        self.X = self.X[mask]
        y = y[mask]

        # convert to 0/1: binding=0, cryptic=1
        self.Ys = torch.argmax(y[:, :2], dim=1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Ys[idx]
    
class Stage1Model(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = EmmaClassifier(input_dim)
        self.net.layer_4 = nn.Linear(LAYER_WIDTH, 2)

    def forward(self, x):
        return self.net(x)


class Stage2Model(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = EmmaClassifier(input_dim)
        self.net.layer_4 = nn.Linear(LAYER_WIDTH, 2)

    def forward(self, x):
        return self.net(x)

# ### STAGE 1 ###
# emb_space = EMB_SPACES[2]
# device = 'cuda' if torch.cuda.is_available() else 'cpu'
# print(f'Running on device {device} ...')
# 
# with open(f'{DATA_PATH}/test_protein_ids.pkl', 'rb') as f:
#     test_protein_ids = pickle.load(f)
# with open(f'{DATA_PATH}/protein_ids.pkl', 'rb') as f:
#     protein_ids = pickle.load(f)
# 
# batch_size = 2048
# stage1_train = Stage1Dataset(train_dataset)
# stage1_test = Stage1Dataset(test_dataset)
# 
# stage1_model = Stage1Model(input_dim=stage1_train.X.shape[1]).to(device)
# 
# optimizer1 = torch.optim.AdamW(stage1_model.parameters(), lr=1e-5)
# criterion1 = nn.CrossEntropyLoss()
# 
# loader1 = DataLoader(stage1_train, batch_size=batch_size, shuffle=True)
# loader2 = DataLoader(stage1_test, batch_size=stage1_test.Ys.shape[0], shuffle=True)
# 
# 
# for epoch in range(epochs):
#     stage1_model.train()
#     for x, y in loader1:
#         x, y = x.to(device), y.to(device)
# 
#         logits = stage1_model(x)
#         loss = criterion1(logits, y)
# 
#         optimizer1.zero_grad()
#         loss.backward()
#         optimizer1.step()
# 
#     stage1_model.eval()
#     for x, y in loader2:
#         x, y = x.to(device), y.to(device)
# 
#         logits = stage1_model(x)
#         loss = criterion1(logits, y)
#         print(f"Stage 1 - Epoch: {epoch} | Test Loss: {loss:.5f}")

### STAGE 2 ###
stage2_train = Stage2Dataset(train_dataset)
stage2_test = Stage2Dataset(test_dataset)
stage2_model = Stage2Model(input_dim=train_dataset.X.shape[1]).to(device)

optimizer2 = torch.optim.AdamW(stage2_model.parameters(), lr=3e-5)
counts = torch.bincount(stage2_train.Ys)
weights = 1.0 / counts.float()
weights = weights / weights.sum()

criterion2 = nn.CrossEntropyLoss(weight=weights.to(device))

from torch.utils.data import WeightedRandomSampler, DataLoader

# Stage2Dataset already defined
stage2_train_dataset = Stage2Dataset(train_dataset)

# Count number of samples per class
y_stage2 = stage2_train_dataset.Ys
class_counts = torch.bincount(y_stage2)
num_classes = len(class_counts)

# Inverse class frequency
class_weights = 1.0 / class_counts.float()

# Assign per-sample weights
sample_weights = class_weights[y_stage2]

# Create weighted sampler
weighted_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

loader1 = DataLoader(stage2_train, batch_size=batch_size, sampler=weighted_sampler)
loader2 = DataLoader(stage2_test, batch_size=stage2_test.Ys.shape[0], shuffle=True)

best_loss = float('inf')
for epoch in range(epochs):
    stage2_model.train()
    for x, y in loader1:
        x, y = x.to(device), y.to(device)

        logits = stage2_model(x)
        loss = criterion2(logits, y)

        optimizer2.zero_grad()
        loss.backward()
        optimizer2.step()

    stage2_model.eval()
    for x, y in loader2:
        x, y = x.to(device), y.to(device)

        logits = stage2_model(x)
        loss = criterion2(logits, y)
        print(f"Stage 2 - Epoch: {epoch} | Test Loss: {loss:.5f}")
    if test_loss < best_loss:
        best_loss = test_loss
        import copy
        best_model = copy.deepcopy(stage2_model.state_dict())

### INFERENCE ### 
stage2_model.load_state_dict(best_model)
stage1_model.eval()
stage2_model.eval()

X_test = test_dataset.Xs.to(device)
y_test = torch.argmax(test_dataset.Ys, dim=1).cpu().numpy()

threshold = 0.1  #

final_preds = []

with torch.no_grad():
    # Stage 1 predictions
    logits1 = stage1_model(X_test)
    probs1 = torch.softmax(logits1, dim=1)

    for i in range(len(X_test)):
        # probability of "binding-related" class (class 1)
        p_binding = probs1[i, 1].item()

        if p_binding < threshold:
            # predict NON-BINDING
            final_preds.append(2)
        else:
            # go to Stage 2
            logits2 = stage2_model(X_test[i].unsqueeze(0))
            pred2 = torch.argmax(logits2, dim=1).item()

            if pred2 == 0:
                final_preds.append(0)  # BINDING
            else:
                final_preds.append(1)  # CRYPTIC

final_preds = np.array(final_preds)
### EVALUATION ###
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(y_test, final_preds)
print("Confusion matrix:")
print(cm)

print(classification_report(y_test, final_preds))

Stage 2 - Epoch: 0 | Test Loss: 0.69283
Stage 2 - Epoch: 1 | Test Loss: 0.69300
Stage 2 - Epoch: 2 | Test Loss: 0.69326
Stage 2 - Epoch: 3 | Test Loss: 0.69341
Stage 2 - Epoch: 4 | Test Loss: 0.69364
Stage 2 - Epoch: 5 | Test Loss: 0.69357
Stage 2 - Epoch: 6 | Test Loss: 0.69369
Stage 2 - Epoch: 7 | Test Loss: 0.69378
Stage 2 - Epoch: 8 | Test Loss: 0.69446
Stage 2 - Epoch: 9 | Test Loss: 0.69558
Stage 2 - Epoch: 10 | Test Loss: 0.69664
Stage 2 - Epoch: 11 | Test Loss: 0.69712
Stage 2 - Epoch: 12 | Test Loss: 0.70197
Stage 2 - Epoch: 13 | Test Loss: 0.70806
Stage 2 - Epoch: 14 | Test Loss: 0.71331
Stage 2 - Epoch: 15 | Test Loss: 0.72270
Stage 2 - Epoch: 16 | Test Loss: 0.73493
Stage 2 - Epoch: 17 | Test Loss: 0.74375
Stage 2 - Epoch: 18 | Test Loss: 0.75728
Stage 2 - Epoch: 19 | Test Loss: 0.77097
Confusion matrix:
[[   30  1420  1300]
 [   47  2248   804]
 [  234 12009 59640]]
              precision    recall  f1-score   support

           0       0.10      0.01      0.02      2750

In [42]:
print(len(stage2_train))

25934
